# Positional Encoding 심층 분석 - 실습 코드 1: RoPE 구현 (PyTorch)

- Tutorial ID: `expand-positional-encoding`
- Tutorial: Positional Encoding 심층 분석
- Section ID: `expand-positional-encoding-code-1`
- Section: 실습 코드 1: RoPE 구현 (PyTorch)

> 이 노트북은 RoPE(Rotary Positional Embedding)를 처음 접하는 분들을 위해 개념 설명, 손으로 따라갈 수 있는 작은 수치 예제, 시각화를 보강한 버전입니다. 새로운 개념이 나올 때마다 바로 아래에서 코드로 같은 내용을 다시 확인할 수 있도록 구성했습니다.

## 코드 읽는 법

이 노트북은 "정답 코드를 한 번 실행해서 결과만 확인"하는 용도가 아니라, **수학적인 개념이 실제 텐서 연산으로 바뀌는 과정을 한 줄씩 추적**하기 위한 실습 노트입니다.

**학습 목표**
1. Query(Q)/Key(K) 벡터에 위치 정보가 "회전"이라는 형태로 어떻게 들어가는지 손으로 계산해서 확인합니다.
2. `RotaryPositionalEmbedding` 클래스가 만드는 `cos`, `sin` 값의 shape과 의미를 추적합니다.
3. RoPE를 적용하면 attention score(내적)가 절대 위치가 아니라 **상대 위치(거리)** 에만 의존한다는 핵심 성질을 직접 수치로 검증합니다.

**읽는 순서**
1. 먼저 작은 2차원 벡터 하나로 "회전시키면 왜 상대 위치만 남는가"를 직관적으로 이해합니다.
2. 차원이 커지면 회전을 "여러 개의 서로 다른 속도"로 나눠서 적용해야 하는 이유를 살펴봅니다.
3. `rotate_half`라는 코드 트릭이 실제로 어떤 숫자를 만들어내는지 작은 벡터로 직접 확인합니다.
4. 위 부품들을 모아 진짜 `RotaryPositionalEmbedding` 클래스를 완성하고, 작은 예제 → 실전 크기 예제 순서로 검증합니다.

**주의할 점**
- 숫자 하나하나를 외우기보다 **"회전 각도가 위치에 비례해서 커진다"**, **"두 벡터를 내적하면 상대 위치만 남는다"** 는 두 가지 큰 그림을 기억하세요.
- 이 노트북은 PyTorch와 matplotlib에 의존하므로 Colab, 로컬 Jupyter, 또는 서버 노트북에서 실행하는 것을 권장합니다. (`pip install torch matplotlib`)

## 0. 들어가며 — Transformer는 왜 "위치"를 따로 알려줘야 할까?

Self-Attention은 입력 토큰들 사이의 관계를 계산할 때 토큰의 **내용(값)** 만 보고 계산합니다. 즉 "이 단어가 몇 번째 단어인지"는 attention 연산 자체에 전혀 반영되지 않습니다.

예를 들어 "나는 너를 사랑해"와 "사랑해 너를 나는"이라는 두 문장을 생각해 봅시다. 단어 집합은 동일하기 때문에, 만약 모델에 순서 정보를 따로 주지 않는다면 Self-Attention은 이 두 문장을 사실상 구분하지 못합니다. 그래서 Transformer 계열 모델은 항상 **위치 정보를 추가로 주입**해야 합니다.

위치 정보를 주는 방법은 크게 두 갈래로 나뉩니다.

- **절대 위치 인코딩 (absolute positional encoding)**: "이 토큰은 3번째 위치다"라는 정보를 토큰 임베딩에 직접 더해주는 방식. 원조 Transformer 논문의 sin/cos positional encoding이나, 학습 가능한 position embedding이 여기에 속합니다.
- **상대 위치 인코딩 (relative positional encoding)**: "이 두 토큰은 3칸 떨어져 있다"는 **거리** 정보를 attention score 계산 과정에 반영하는 방식. 이번 실습의 주제인 RoPE(Rotary Positional Embedding)가 대표적인 예입니다.

RoPE의 핵심 아이디어를 한 문장으로 요약하면 다음과 같습니다.

> **Query/Key 벡터를 "위치만큼" 회전시키면, 두 벡터의 내적(attention score)에는 자연스럽게 두 위치의 차이(상대 거리)만 남는다.**

왜 그런지 아주 작은 2차원 벡터 예제로 직접 확인해 보겠습니다.

## 1. 핵심 아이디어 — 2차원 회전으로 살펴보기

가장 단순한 2차원 $(x, y)$ 벡터를 각도 $\theta$만큼 반시계 방향으로 회전시키는 행렬은 다음과 같습니다.

$$
R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}
$$

회전 행렬에는 아주 중요한 성질이 하나 있습니다. 벡터 $q$를 각도 $m\theta$만큼 돌리고 벡터 $k$를 각도 $n\theta$만큼 돌린 다음 두 벡터를 내적하면,

$$
\big(R(m\theta)\,q\big)^{\top} \big(R(n\theta)\,k\big) \;=\; q^{\top} R\big((n-m)\theta\big)\, k
$$

가 됩니다. 즉 **결과는 $m$, $n$ 각각의 값이 아니라 오직 그 차이 $(n-m)$에만 의존**합니다. ($m=2, n=5$이든 $m=10, n=13$이든, 차이가 똑같이 3이면 결과도 같습니다.)

RoPE는 바로 이 성질을 이용합니다.
- 위치 $m$에 있는 query 벡터는 $m\theta$만큼 회전시키고,
- 위치 $n$에 있는 key 벡터는 $n\theta$만큼 회전시킨 뒤,
- 두 벡터를 내적해서 attention score를 구하면,

그 score에는 "몇 번째 토큰인지"가 아니라 "몇 칸 떨어져 있는지"만 자연스럽게 반영됩니다.

말로만 들으면 추상적이니, 아래 코드로 진짜 그런지 직접 확인해 보겠습니다.

In [ ]:
import torch
import torch.nn as nn  # 잠시 후 RotaryPositionalEmbedding 클래스를 만들 때 사용합니다
import math            # 이 셀에서는 math.cos / math.sin으로 "스칼라" 회전을 직접 계산해 봅니다

torch.manual_seed(0)  # 실행할 때마다 같은 난수가 나오도록 고정 (재현성을 위함)

def rotate_2d_simple(vec, angle):
    # 아주 단순화된 2D 회전 함수입니다. 실제 RoPE 구현(뒤에서 만들 rotate_half 기반)이 아니라
    # "회전이 어떤 효과를 내는지" 감을 잡기 위한 임시(toy) 함수라고 생각하면 됩니다.
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    x, y = vec[0].item(), vec[1].item()
    new_x = x * cos_a - y * sin_a   # 회전행렬 1행: [cos, -sin] · [x, y]
    new_y = x * sin_a + y * cos_a   # 회전행렬 2행: [sin,  cos] · [x, y]
    return torch.tensor([new_x, new_y])

theta = 0.4  # 한 칸(position) 이동할 때마다 추가로 회전하는 각도(라디안). RoPE에서는 이 값이 차원마다 달라집니다 (2장에서 설명).

q = torch.tensor([1.0, 0.5])   # "질문(query)"의 내용 벡터 — 위치와 무관하게 고정되어 있다고 가정합니다.
k = torch.tensor([0.3, -0.8])  # "키(key)"의 내용 벡터 — 역시 위치와 무관하게 고정.

def attention_score(m, n):
    # 위치 m에 있는 q와 위치 n에 있는 k 사이의 (회전 적용 후) attention score
    q_m = rotate_2d_simple(q, m * theta)  # q를 위치 m만큼 회전
    k_n = rotate_2d_simple(k, n * theta)  # k를 위치 n만큼 회전
    return torch.dot(q_m, k_n).item()     # 두 회전된 벡터의 내적 = attention score

print("=== 상대 거리(n - m)가 똑같이 3인 경우들 ===")
print(f"m=2,  n=5  (거리 3) -> score = {attention_score(2, 5):.6f}")
print(f"m=10, n=13 (거리 3) -> score = {attention_score(10, 13):.6f}")
print(f"m=0,  n=3  (거리 3) -> score = {attention_score(0, 3):.6f}")

print("\n=== 상대 거리가 다른 경우들 (비교용) ===")
print(f"m=2, n=4 (거리 2) -> score = {attention_score(2, 4):.6f}")
print(f"m=2, n=6 (거리 4) -> score = {attention_score(2, 6):.6f}")

### 실행 결과 해석

`m=2,n=5`, `m=10,n=13`, `m=0,n=3`처럼 **거리가 3으로 같은 세 경우는 모두 약 0.8492로 거의 동일한 score**가 나옵니다. (소수점 마지막 자리의 아주 작은 차이는 부동소수점 연산 오차일 뿐입니다.) 반면 거리가 2이거나 4인 경우는 전혀 다른 값이 나옵니다.

즉 `q`, `k`의 내용은 그대로 둔 채 위치 $m, n$ 자체를 바꾸더라도, **차이 $(n-m)$만 같으면 attention score가 똑같다**는 것을 직접 확인했습니다. 이것이 RoPE가 "상대 위치 인코딩"으로 분류되는 이유입니다.

이제 이 아이디어를 실제 모델에서 쓰는 고차원(예: 64차원, 128차원) 벡터로 확장해 보겠습니다.

## 2. 차원이 커지면? — 여러 개의 "회전 속도"를 함께 쓰기

실제 Transformer의 head_dim은 보통 64나 128처럼 짝수의 큰 숫자입니다. 이렇게 많은 차원을 앞서 본 2D 회전 하나로 처리할 수는 없습니다. 그래서 RoPE는 $d$차원 벡터를 **$d/2$개의 2차원 쌍(pair)** 으로 나눈 뒤, **각 쌍마다 서로 다른 회전 속도(주파수)** 를 적용합니다.

비유하자면 아날로그 시계를 떠올려 보세요.

- **초침**은 아주 빠르게 돌아서 "지금이 몇 초인지"처럼 아주 미세한 차이를 구분합니다.
- **시침**은 아주 느리게 돌아서 "지금이 대략 몇 시인지"처럼 큰 그림을 구분합니다.

초침 하나만으로는 "지금이 1시인지 2시인지"를 알 수 없고, 시침 하나만으로는 "지금이 1시 1분인지 1시 2분인지"를 알 수 없습니다. **여러 속도의 바늘을 함께 봐야 정확한 시각을 알 수 있는 것처럼**, RoPE도 빠르게 도는 쌍(가까운 거리 차이를 잘 구분)과 느리게 도는 쌍(먼 거리까지 표현 가능)을 함께 사용해서 다양한 거리 스케일을 표현합니다.

각 쌍 $i$ ($i = 0, 2, 4, \dots, d-2$, 즉 짝수 인덱스로 $d/2$개) 의 회전 속도(각주파수) $\theta_i$는 다음 식으로 정해집니다.

$$
\theta_i = \text{base}^{-i/d} = \frac{1}{\text{base}^{\,i/d}}, \qquad \text{base}=10000 \ \text{(기본값)}
$$

- $i=0$일 때 $\theta_0 = 1$ → 가장 빠르게 회전 (초침 역할)
- $i$가 커질수록 $\theta_i$는 지수적으로 작아짐 → 점점 느리게 회전 (시침 역할)

코드로 직접 계산해서 숫자로 확인해 보겠습니다.

In [ ]:
# 작은 예제를 위해 dim=8 (즉 4개의 회전 쌍)을 사용합니다. 실제 모델에서는 64, 128 등을 씁니다.
dim = 8
base = 10000

# torch.arange(0, dim, 2) -> [0, 2, 4, 6] : 짝수 인덱스만 뽑습니다 (각 쌍의 "대표 인덱스")
even_idx = torch.arange(0, dim, 2).float()
print("짝수 인덱스 i     :", even_idx.tolist())
print("i / dim           :", (even_idx / dim).tolist())

# theta_i = base^(-i/dim) = 1 / base^(i/dim)
inv_freq = 1.0 / (base ** (even_idx / dim))
print("회전 속도 theta_i  :", [round(v, 4) for v in inv_freq.tolist()])

dim=8일 때는 숫자가 깔끔하게 떨어집니다: `theta = [1.0, 0.1, 0.01, 0.001]`.

- 0번째 쌍은 한 칸 이동할 때마다 1라디안(약 57˚)씩 빠르게 회전합니다 → "초침"
- 마지막(3번째) 쌍은 한 칸 이동할 때마다 0.001라디안씩만 회전합니다 → "시침" (1000칸을 가야 1라디안 회전)

이제 위치 $t = 0, 1, 2, \dots$ 가 늘어날 때 각 쌍의 회전각이 $t \times \theta_i$로 커진다는 것을, 그리고 이 회전 속도 차이를 그래프로 직접 확인해 보겠습니다.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(even_idx.tolist(), inv_freq.tolist(), 'o-', color='#2563eb')
ax.set_yscale('log')  # 값이 1, 0.1, 0.01, 0.001처럼 자릿수 단위로 줄어들기 때문에 로그 스케일로 봐야 잘 보입니다
ax.set_xlabel('dimension pair index i (0, 2, 4, 6)')
ax.set_ylabel('theta_i  (log scale)')
ax.set_title(f'RoPE rotation speed per pair  (dim={dim}, base={base})')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

차원 인덱스가 커질수록 회전 속도가 기하급수적으로 줄어드는 것이 한눈에 보입니다. 이번에는 위치가 0부터 늘어날 때 "빠른 쌍"과 "느린 쌍"이 실제로 평면 위에서 얼마나 다르게 움직이는지 시각화해 보겠습니다. 한 쌍 $(x_i, x_{i+d/2})$을 2차원 평면의 한 점 $\big(\cos(t\theta_i),\ \sin(t\theta_i)\big)$이라고 생각하면 됩니다.

In [ ]:
positions = torch.arange(0, 25)  # 위치 0~24까지 살펴봅니다

fig, ax = plt.subplots(figsize=(5.5, 5.5))

circle = plt.Circle((0, 0), 1.0, fill=False, color='lightgray', linestyle='--')  # 참고용 단위원
ax.add_patch(circle)

colors = ['#dc2626', '#ea580c', '#16a34a', '#2563eb']  # 빠른 것부터 느린 것 순서

for i, (c, theta_i) in enumerate(zip(colors, inv_freq.tolist())):
    angles = positions.float() * theta_i        # 위치 t에서의 회전각 = t * theta_i
    xs = torch.cos(angles)
    ys = torch.sin(angles)
    ax.plot(xs, ys, 'o-', color=c, markersize=4, alpha=0.8,
            label=f'pair i={i*2}  (theta={theta_i:.3f} rad/step)')
    ax.plot(xs[0], ys[0], '*', color=c, markersize=14)  # 위치 0(시작점)을 별표로 표시

ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_title('How each pair rotates as position goes 0 -> 24\n(star = position 0)')
ax.legend(fontsize=8, loc='upper right')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

빨간색($i=0$)은 24칸을 이동하는 동안 원을 여러 바퀴나 돌았지만, 파란색($i=6$)은 거의 출발점(★)에서 움직이지 않았습니다. 이렇게 서로 다른 속도로 도는 바늘들을 모아 두면, 가까운 위치 차이는 빠른 바늘이, 먼 위치 차이는 느린 바늘이 나누어 표현하게 됩니다. 이것이 RoPE가 짧은 거리와 긴 거리를 동시에 표현할 수 있는 이유입니다.

## 3. `rotate_half` — 코드로 "회전 짝(pair)"을 만드는 트릭

지금까지는 회전시킬 두 차원의 짝(pair)을 (0번, 4번)처럼 미리 정해서 설명했습니다. 실제 구현에서는 이 "짝을 만드는" 과정이 `rotate_half`라는 함수 하나로 처리됩니다. 이름 그대로 벡터를 반(half)으로 나눠서 회전(rotate)에 필요한 짝을 만드는 함수인데, 정확히 어떤 숫자를 만드는지 작은 벡터로 직접 추적해 보겠습니다.

핵심만 먼저 말씀드리면, 차원을 **앞쪽 절반**과 **뒤쪽 절반**으로 나눈 뒤 $i$번째 차원과 $(i + d/2)$번째 차원을 한 쌍으로 묶습니다. $d=8$이라면 (0,4), (1,5), (2,6), (3,7)이 네 개의 쌍이 됩니다.

> **참고**: RoPE 원 논문은 (0,1), (2,3)처럼 "바로 옆자리"끼리 짝짓는 방식을 사용하지만, 실제로 LLaMA 등 널리 쓰이는 구현체들은 지금처럼 "절반씩 나눠 짝짓는" 방식을 사용합니다. 차원의 순서만 다를 뿐, 두 방식은 수학적으로 동등합니다 (가중치 행렬이 어차피 학습되기 때문에 차원을 어떤 순서로 짝짓든 모델 성능에는 차이가 없습니다).

In [ ]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)          # 마지막 차원을 정확히 절반으로 자릅니다
    return torch.cat((-x2, x1), dim=-1)  # [-뒤쪽 절반, 앞쪽 절반] 순서로 다시 이어붙입니다

x = torch.tensor([1., 2., 3., 4., 5., 6., 7., 8.])  # dim=8짜리 예시 벡터. 값을 1~8로 줘서 손으로 추적하기 쉽게 했습니다.

x1, x2 = x.chunk(2, dim=-1)
print("원본 x           :", x.tolist())
print("앞쪽 절반 x1      :", x1.tolist())  # [1, 2, 3, 4] 가 나와야 함
print("뒤쪽 절반 x2      :", x2.tolist())  # [5, 6, 7, 8] 가 나와야 함
print("rotate_half(x)   :", rotate_half(x).tolist())  # [-5, -6, -7, -8, 1, 2, 3, 4] 가 나와야 함

결과는 `[-5, -6, -7, -8, 1, 2, 3, 4]` 입니다. 즉 `rotate_half`는 "0번 차원의 짝꿍은 4번 차원(부호 반전), 4번 차원의 짝꿍은 0번 차원"이라는 관계를 한 번에 만들어 줍니다.

이렇게 만든 `rotate_half(x)`에 $\sin$을 곱하고, 원래의 `x`에는 $\cos$을 곱해서 더하면 — 바로 다음 절에서 확인하겠지만 — 정확히 2D 회전 공식이 완성됩니다.

## 4. `cos`·`sin` + `rotate_half` = 진짜 2D 회전

이제 모든 조각을 연결해 보겠습니다. RoPE를 적용하는 핵심 줄은 다음과 같습니다.

$$
x_{\text{rotated}} \;=\; x \odot \cos(\Theta) \;+\; \text{rotate\_half}(x) \odot \sin(\Theta)
$$

($\odot$는 원소별 곱셈, $\Theta$는 위치 $t$와 각 쌍의 속도 $\theta_i$로 만든 각도들의 모음입니다.)

이 식이 1장에서 본 회전행렬과 정말 같은지, 차원 $(i,\ i+d/2)$ 쌍에 대해 직접 풀어보겠습니다.

$$
x'_i = x_i\cos(t\theta_i) - x_{i+d/2}\sin(t\theta_i)
$$

$$
x'_{i+d/2} = x_i\sin(t\theta_i) + x_{i+d/2}\cos(t\theta_i)
$$

`rotate_half(x)`의 $i$번째 값이 $-x_{i+d/2}$이고 $(i+d/2)$번째 값이 $x_i$였다는 것을 떠올리면(3장 참고), 위 두 식이 정확히 "$x \odot \cos + \text{rotate\_half}(x) \odot \sin$" 한 줄에서 그대로 나온다는 것을 알 수 있습니다.

즉 `cos`, `sin`, `rotate_half`라는 세 가지 재료만으로, 반복문 없이 $d/2$개의 2D 회전을 한 번에 벡터 연산으로 처리하는 것이 RoPE 구현의 핵심 트릭입니다.

## 5. 메인 구현 — `RotaryPositionalEmbedding` (오류 수정 버전)

지금까지 확인한 세 가지 재료(주파수 $\theta_i$, `rotate_half`, `cos`·`sin` 공식)를 모아 실제 모듈로 정리하겠습니다.

> **참고**: 원래 노트북 코드는 `__init__`과 `forward`의 들여쓰기가 `def` 줄과 같은 깊이로 되어 있어 `IndentationError`가 발생하고 실행되지 않는 상태였습니다. Python은 들여쓰기 자체가 문법의 일부라서, 메서드 본문은 `def` 줄보다 반드시 한 단계 더 깊게 들여써야 합니다. 아래 코드는 들여쓰기를 표준 4칸으로 맞추고, 각 줄에 설명 주석을 추가했습니다.

In [ ]:
import torch
import torch.nn as nn

# RotaryPositionalEmbedding
# -----------------------------------------------------------------
# 이 모듈이 하는 일: 위치 t에 대한 cos(t*theta_i), sin(t*theta_i) "표"를 미리 계산해서 돌려줍니다.
# 이 모듈 자체는 q, k를 회전시키지 않습니다 — 회전에 필요한 cos/sin 값만 만들어 줄 뿐이고,
# 실제 회전(곱하고 더하는 연산)은 아래 apply_rotary_pos_emb 함수가 담당합니다.
class RotaryPositionalEmbedding(nn.Module):

    def __init__(self, dim, max_seq_len=8192, base=10000):
        # dim        : head_dim (attention head 하나가 갖는 벡터 차원). 반드시 짝수여야 합니다.
        # max_seq_len: 참고용으로 저장해 두는 값 (forward에서 seq_len을 직접 받으므로 강제하지는 않음)
        # base       : 회전 속도를 정하는 기준 상수. 클수록 가장 느린 쌍의 회전이 더 완만해집니다. (기본 10000)
        super().__init__()  # nn.Module의 초기화를 먼저 호출 (파라미터/버퍼 등록 기능을 쓰려면 필수)

        # theta_i = base^(-i/dim),  i = 0, 2, 4, ..., dim-2  -> 총 dim/2개의 값 (2장에서 만든 식과 동일)
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

        # register_buffer: 학습되는 파라미터(parameter)는 아니지만, 모델과 함께 device(GPU 등)로 옮겨지고
        # state_dict에는 저장되도록 등록하는 방법입니다. (옵티마이저가 값을 업데이트하지는 않습니다)
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len

    def forward(self, x, seq_len=None):
        # x       : (batch, seq_len, num_heads, head_dim) 모양의 텐서. 여기서는 device 정보를 얻으려고만 사용합니다.
        # seq_len : 위치 0, 1, ..., seq_len-1 에 대한 cos/sin을 만듭니다. 없으면 x의 seq 길이를 사용합니다.
        seq_len = seq_len or x.shape[1]

        # t = [0, 1, 2, ..., seq_len-1]  (위치 인덱스). type_as로 inv_freq와 dtype/device를 맞춥니다.
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)

        # einsum('i,j->ij', t, inv_freq): t(seq_len,)와 inv_freq(dim/2,)의 모든 조합을 곱해서
        # (seq_len, dim/2) 모양의 "위치별 회전각" 표를 만듭니다.  freqs[pos, i] = pos * theta_i
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)

        # rotate_half가 [-뒤쪽 절반, 앞쪽 절반] 순서로 다루므로 (3장 참고),
        # cos/sin도 앞/뒤 절반에 같은 theta_i 값을 복제해 둡니다. (한 쌍은 같은 회전각을 공유하기 때문)
        emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, dim)

        cos = emb.cos()  # (seq_len, dim)
        sin = emb.sin()  # (seq_len, dim)
        return cos, sin


# rotate_half: 3장에서 작은 벡터로 직접 동작을 확인했던 바로 그 함수입니다.
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


# apply_rotary_pos_emb: q, k에 실제로 회전을 적용하는 함수
#   q, k     : (batch, seq_len, num_heads, head_dim)
#   cos, sin : (seq_len, head_dim)  <- RotaryPositionalEmbedding.forward()가 만든 값
def apply_rotary_pos_emb(q, k, cos, sin):
    # cos/sin은 (seq, dim)이지만 q/k는 (batch, seq, heads, dim)이므로,
    # batch 차원(0번 자리)과 heads 차원(2번 자리)에 크기 1인 차원을 끼워 넣어 broadcasting되게 만듭니다.
    cos = cos.unsqueeze(0).unsqueeze(2)  # (seq, dim) -> (1, seq, 1, dim)
    sin = sin.unsqueeze(0).unsqueeze(2)

    # 4장에서 본 바로 그 공식: x_rotated = x*cos + rotate_half(x)*sin
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
torch.manual_seed(0)

seq_len = 4   # 위치 0,1,2,3 네 개만 살펴봅니다 (dim은 위에서 쓰던 8을 그대로 사용)

rope = RotaryPositionalEmbedding(dim=dim)

# 실전에서는 q,k가 (batch, seq, heads, dim)이지만, 지금은 batch=1, heads=1로 단순화합니다.
q = torch.randn(1, seq_len, 1, dim)
k = torch.randn(1, seq_len, 1, dim)

cos, sin = rope(q, seq_len=seq_len)
print("cos shape:", tuple(cos.shape), " sin shape:", tuple(sin.shape))  # (4, 8), (4, 8) 이어야 함

print("\n위치 0의 cos (회전각이 전부 0 -> cos(0)=1 이므로 전부 1이어야 함):")
print([round(v, 4) for v in cos[0].tolist()])

print("\n위치 1의 cos, sin (theta = [1, 0.1, 0.01, 0.001]이 그대로 보임):")
print("cos[1] =", [round(v, 4) for v in cos[1].tolist()])
print("sin[1] =", [round(v, 4) for v in sin[1].tolist()])

q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
print("\nq shape:", tuple(q.shape), "-> 회전 후 q_rot shape:", tuple(q_rot.shape), "(shape은 그대로, 값만 회전)")

# 위치 0은 회전각이 0이므로, 회전시켜도 원래 벡터와 같아야 합니다.
print("위치 0에서 q == q_rot ?", torch.allclose(q[0, 0, 0], q_rot[0, 0, 0], atol=1e-6))

In [ ]:
# 검증: apply_rotary_pos_emb의 결과가, 1장에서 손으로 계산한 2x2 회전행렬과 정말 같은지 직접 비교합니다.
position = 2
pair_index = 0  # 0번째 쌍 -> 원래 차원으로는 (0번, 4번) 차원

theta_i = rope.inv_freq[pair_index].item()
angle = position * theta_i
cos_a, sin_a = math.cos(angle), math.sin(angle)

v0 = q[0, position, 0, 0].item()  # 회전 전 0번 차원 값
v4 = q[0, position, 0, 4].item()  # 회전 전 4번 차원 값 (0번의 짝꿍)

# 1장의 2x2 회전 공식을 그대로 손으로 적용
manual_v0 = v0 * cos_a - v4 * sin_a
manual_v4 = v0 * sin_a + v4 * cos_a

print(f"[위치={position}, 쌍 index={pair_index}  (원래 차원 0번, 4번)]")
print(f"apply_rotary_pos_emb 결과 : {q_rot[0, position, 0, 0].item():.6f}, {q_rot[0, position, 0, 4].item():.6f}")
print(f"손으로 계산한 2x2 회전값  : {manual_v0:.6f}, {manual_v4:.6f}")
print("두 값이 (거의) 같다면, cos/sin + rotate_half 조합이 진짜 2D 회전과 동일하다는 뜻입니다.")

## 6. 다시 한번: "상대 위치"만 영향을 준다는 것을 8차원에서도 확인하기

1장에서는 2차원 벡터 하나로 이 성질을 보였습니다. 이번에는 실제 `RotaryPositionalEmbedding` 클래스가 만든 cos/sin 표를 그대로 사용해서, dim=8인 벡터에서도 똑같은 성질이 성립하는지 확인합니다.

방법은 동일합니다.
1. 위치와 무관한 query 내용 벡터 `q_content`, key 내용 벡터 `k_content`를 하나씩 고정합니다.
2. 여러 개의 $(m, n)$ 위치 쌍에 대해 회전을 적용한 뒤 내적(attention score)을 계산합니다.
3. **$n-m$이 같은 쌍은 score도 같고, 다르면 score도 다른지** 확인합니다.

이 성질이 중요한 이유는, 모델이 "이 토큰이 절대적으로 몇 번째인지"가 아니라 "내가 보려는 토큰이 나로부터 얼마나 떨어져 있는지"에 집중하게 해주기 때문입니다. 그 덕분에 학습 때 보지 못한 더 긴 시퀀스(더 큰 절대 위치)에 대해서도 비교적 잘 일반화하는 경향이 있습니다.

In [ ]:
torch.manual_seed(42)

MAX_POS = 20
# (1, MAX_POS, 1, dim) 모양의 더미 텐서를 넣어 위치 0~19에 대한 cos/sin 표를 미리 만들어 둡니다.
cos_table, sin_table = rope(torch.zeros(1, MAX_POS, 1, dim), seq_len=MAX_POS)

q_content = torch.randn(dim)  # 위치와 무관한 query 내용 (하나의 head_dim 벡터)
k_content = torch.randn(dim)  # 위치와 무관한 key 내용

def rotated_score(m, n):
    # 배치/헤드 차원 없이 단일 벡터(dim,)에 직접 공식을 적용합니다.
    # rotate_half는 "마지막 차원" 기준으로 동작하므로 1차원 벡터에도 그대로 재사용할 수 있습니다.
    q_m = q_content * cos_table[m] + rotate_half(q_content) * sin_table[m]
    k_n = k_content * cos_table[n] + rotate_half(k_content) * sin_table[n]
    return torch.dot(q_m, k_n).item()

print("--- 상대 거리 (n - m) = 3 으로 동일한 여러 (m, n) ---")
for m, n in [(2, 5), (10, 13), (0, 3)]:
    print(f"m={m:2d}, n={n:2d}  ->  score = {rotated_score(m, n): .6f}")

print("\n--- 상대 거리가 서로 다른 경우들 (비교용) ---")
for m, n in [(2, 4), (2, 6), (2, 7)]:
    print(f"m={m:2d}, n={n:2d}  (거리 {n-m})  ->  score = {rotated_score(m, n): .6f}")

### 실행 결과 해석

거리($n-m$)가 3으로 같은 세 쌍은 score가 약 **-1.1043**으로 모두 거의 동일하게 나오고, 거리가 다른 쌍들(2, 4, 5)은 각각 -2.4374, -0.6524, -1.4806으로 서로 다르게 나옵니다. (직접 실행한 값이 이 숫자와 같다면 잘 따라온 것입니다. seed를 바꾸면 `q_content`/`k_content` 자체가 달라지므로 절대값은 달라지지만, "거리가 같으면 score도 같다"는 패턴은 항상 유지됩니다.)

dim=8이라는 작은 예제에서도, 실제 모델에서 쓰는 큰 차원에서도 이 성질은 동일하게 성립합니다. 마지막으로 노트북 맨 처음에 있던 것과 같은 "실전 크기" 예제로 전체 파이프라인을 다시 확인해 보겠습니다.

## 7. 실전 크기로 최종 확인하기

지금까지 만든 `RotaryPositionalEmbedding`, `rotate_half`, `apply_rotary_pos_emb`는 차원 크기에 관계없이 그대로 동작합니다. 이제 실제 모델에서 흔히 쓰는 크기 — batch=2, seq_len=10, num_heads=8, head_dim=64 — 로 다시 실행해서 shape이 기대한 대로 나오는지, 그리고 회전이 벡터의 길이(norm)는 바꾸지 않는다는 사실(회전은 방향만 바꾸고 크기는 보존합니다)도 함께 확인합니다.

In [ ]:
rope_big = RotaryPositionalEmbedding(dim=64)

q = torch.randn(2, 10, 8, 64)  # batch=2, seq=10, heads=8, head_dim=64
k = torch.randn(2, 10, 8, 64)
print("회전 전 q shape (batch, seq, heads, head_dim):", tuple(q.shape))

cos, sin = rope_big(q, seq_len=10)
print("cos shape (seq, head_dim):", tuple(cos.shape))  # (10, 64)

q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
print(f"RoPE applied: q={tuple(q_rot.shape)}, k={tuple(k_rot.shape)}")  # shape은 그대로 (2, 10, 8, 64)

# 회전은 벡터의 길이를 바꾸지 않는 변환입니다 (방향만 바뀌고 크기는 그대로). 정말 그런지 확인:
norm_before = q.norm(dim=-1)
norm_after = q_rot.norm(dim=-1)
print("회전 전/후 벡터 길이(norm)가 보존되는가?", torch.allclose(norm_before, norm_after, atol=1e-4))

## 마무리

이번 실습에서 확인한 핵심 내용을 정리하면 다음과 같습니다.

1. **회전은 상대 위치만 남긴다**: 벡터를 위치에 비례한 각도로 회전시키면, 두 벡터의 내적에는 절대 위치가 아니라 두 위치의 차이(거리)만 반영됩니다.
2. **여러 속도의 회전을 함께 쓴다**: head_dim을 $d/2$개의 2D 쌍으로 나누고, 쌍마다 다른 속도 $\theta_i = \text{base}^{-i/d}$로 회전시켜 짧은 거리부터 긴 거리까지 폭넓게 표현합니다.
3. **`rotate_half` + `cos`/`sin`**: 반복문 없이 모든 쌍의 2D 회전을 한 번에 벡터 연산으로 처리하는 구현 트릭입니다.
4. 구현은 차원 크기와 무관하게 동작하며, 회전이라는 연산의 성질상 벡터의 길이(norm)는 바뀌지 않고 방향만 바뀝니다.

**더 알아보면 좋은 주제**
- RoPE는 GPT-NeoX, LLaMA, PaLM 등 다양한 LLM에서 널리 사용되는 위치 인코딩 방식입니다.
- 학습 때보다 훨씬 긴 시퀀스를 다뤄야 할 때를 위해 `base` 값을 조정하는 **NTK-aware scaling**, 위치 인덱스 자체를 스케일링하는 **Position Interpolation(Linear scaling)** 같은 컨텍스트 길이 확장 기법들도 함께 찾아보면 좋습니다.
- RoPE와 비교되는 또 다른 상대 위치 인코딩 방식으로, attention score에 거리 비례 페널티를 직접 빼주는 **ALiBi**도 살펴볼 만합니다.

이 RoPE 모듈을 실제 Multi-Head Attention 안에 통합해 보거나, `base`나 `dim` 값을 바꿔가며 회전 속도가 어떻게 달라지는지 직접 실험해 보면 이해가 한층 더 깊어질 것입니다.